In [1]:
import spacy
import glob
import pandas as pd

In [2]:
## build nlp pipeline (a function will tokenize, parse and ner for us)
nlp = spacy.load("en_core_web_sm")

In [3]:
import en_core_web_trf

In [4]:
nlp=en_core_web_trf.load()

In [5]:
! python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 19.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [6]:
breeders_df = pd.read_excel("Class A Breeders.xlsx", sheet_name="pivot without dates", skiprows=2)
breeders_df

,Row Labels,Sum of critical + direct
0,Envigo RMS LLC,64.0
1,DANIEL GINGERICH,51.0
2,HENRY SOMMERS,24.0
3,CORY MINCEY,22.0
4,Wilma Jinson,20.0
...,...,...
4640,Backwood Pets LLC,0.0
4641,Cathy Sanders,0.0
4642,Bailey LLC,0.0
4643,BAPTIST RIDGE DOBERMANS INC,0.0


In [7]:
def tokenize_text(text):
    '''
    takes each row label, reads it, and pushes through NLP pipeline
    para1= single file
    
    '''
    return nlp(str(text))

In [8]:
breeders_df["doc"] = breeders_df["Row Labels"].apply(tokenize_text)
breeders_df

,Row Labels,Sum of critical + direct,doc
0,Envigo RMS LLC,64.0,"(Envigo, RMS, LLC)"
1,DANIEL GINGERICH,51.0,"(DANIEL, , GINGERICH)"
2,HENRY SOMMERS,24.0,"(HENRY, SOMMERS)"
3,CORY MINCEY,22.0,"(CORY, MINCEY)"
4,Wilma Jinson,20.0,"(Wilma, Jinson)"
...,...,...,...
4640,Backwood Pets LLC,0.0,"(Backwood, Pets, LLC)"
4641,Cathy Sanders,0.0,"(Cathy, Sanders)"
4642,Bailey LLC,0.0,"(Bailey, LLC)"
4643,BAPTIST RIDGE DOBERMANS INC,0.0,"(BAPTIST, RIDGE, DOBERMANS, INC)"


In [13]:
def get_label(doc):
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG"]:
            return ent.label_
    return "UNKNOWN"


In [15]:
breeders_df["entity_type"] = breeders_df["doc"].apply(get_label)
breeders_df

,Row Labels,Sum of critical + direct,doc,entity_type
0,Envigo RMS LLC,64.0,"(Envigo, RMS, LLC)",ORG
1,DANIEL GINGERICH,51.0,"(DANIEL, , GINGERICH)",PERSON
2,HENRY SOMMERS,24.0,"(HENRY, SOMMERS)",PERSON
3,CORY MINCEY,22.0,"(CORY, MINCEY)",UNKNOWN
4,Wilma Jinson,20.0,"(Wilma, Jinson)",PERSON
...,...,...,...,...
4640,Backwood Pets LLC,0.0,"(Backwood, Pets, LLC)",ORG
4641,Cathy Sanders,0.0,"(Cathy, Sanders)",PERSON
4642,Bailey LLC,0.0,"(Bailey, LLC)",ORG
4643,BAPTIST RIDGE DOBERMANS INC,0.0,"(BAPTIST, RIDGE, DOBERMANS, INC)",ORG


In [17]:
breeders_df.sample(20)

,Row Labels,Sum of critical + direct,doc,entity_type
3285,HIDDEN NOOK KENNELS LLC,0.0,"(HIDDEN, NOOK, KENNELS, LLC)",ORG
904,Toba Miller,0.0,"(Toba, Miller)",PERSON
3133,"Glenda Grove, Eric Grove, Adam Grove",0.0,"(Glenda, Grove, ,, Eric, Grove, ,, Adam, Grove)",PERSON
1031,Robert Braun Sara Braun,0.0,"(Robert, Braun, Sara, Braun)",PERSON
2125,LEROY SCHLABACH,0.0,"(LEROY, SCHLABACH)",PERSON
3208,Edwin Z Leid Ruth M Leid Ervin H Leid,0.0,"(Edwin, Z, Leid, Ruth, M, Leid, Ervin, H, Leid)",PERSON
1469,Michael Kreger,0.0,"(Michael, Kreger)",PERSON
2097,Lynn Long,0.0,"(Lynn, Long)",PERSON
3318,Elmer A Miller,0.0,"(Elmer, A, Miller)",PERSON
1465,ORA BONTRAGER SARA BONTRAGER,0.0,"(ORA, BONTRAGER, SARA, BONTRAGER)",PERSON


In [27]:
breeders_df1= breeders_df[["Row Labels", "Sum of critical + direct", "entity_type"]]
breeders_df1

,Row Labels,Sum of critical + direct,entity_type
0,Envigo RMS LLC,64.0,ORG
1,DANIEL GINGERICH,51.0,PERSON
2,HENRY SOMMERS,24.0,PERSON
3,CORY MINCEY,22.0,UNKNOWN
4,Wilma Jinson,20.0,PERSON
...,...,...,...
4640,Backwood Pets LLC,0.0,ORG
4641,Cathy Sanders,0.0,PERSON
4642,Bailey LLC,0.0,ORG
4643,BAPTIST RIDGE DOBERMANS INC,0.0,ORG


In [43]:
df_org_breeders= breeders_df1.query("entity_type=='ORG'")
df_org_breeders

,Row Labels,Sum of critical + direct,entity_type
0,Envigo RMS LLC,64.0,ORG
34,SANDSTONE VALLEY LLC,6.0,ORG
92,LUV MY PUP LLC,4.0,ORG
114,Cowtown Frenchies and Bluebonnet Bulldogs LLC,4.0,ORG
126,CAPTIVA KENNEL INC.,4.0,ORG
...,...,...,...
4634,B&R Puppies LLC,0.0,ORG
4636,Back Road Pets LLC,0.0,ORG
4640,Backwood Pets LLC,0.0,ORG
4642,Bailey LLC,0.0,ORG


In [45]:
df_individual_breeders= breeders_df1.query("entity_type=='PERSON'")
df_individual_breeders

,Row Labels,Sum of critical + direct,entity_type
1,DANIEL GINGERICH,51.0,PERSON
2,HENRY SOMMERS,24.0,PERSON
4,Wilma Jinson,20.0,PERSON
5,DELORIS RICHARDS DICK RICHARDS,20.0,PERSON
6,Gary Felts,18.0,PERSON
...,...,...,...
4633,CATHERINE C. FRANCKA,0.0,PERSON
4635,Catherine Rexwinkle,0.0,PERSON
4637,Catherine Stauffer,0.0,PERSON
4639,Cathie Atchison,0.0,PERSON


In [51]:
df_org_breeders.to_csv('Class A dog breeder organizations.csv', index=False)
df_individual_breeders.to_csv('Class A individual dog breeders.csv', index=False)